In [1]:
#ライブラリ読み込み
import pandas as pd
import numpy as np
import optuna


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#統計＆土地利用データ読み込み
df = pd.read_csv("~/Desktop/趣味/CS2020_with_landuse_features.csv")

data = df.copy()

print(data.head())
print(data.shape)
print(data.columns)

        地域  TFR2022  R2020_log10(10万人あたり人口密度)  J2020労働力人口女性割合  \
0  北海道 札幌市     1.09                  3.652768        0.471028   
1  北海道 函館市     1.18                  3.308714        0.470285   
2  北海道 小樽市     1.12                  3.142921        0.475134   
3  北海道 旭川市     1.29                  2.971137        0.466380   
4  北海道 室蘭市     1.38                  3.278342        0.414711   

   J2020第1次産業就業者比率【％】  T2020徒歩だけ  T2020鉄道・電車  T2020乗合バス  T2020勤め先・学校のバス  \
0                0.47   0.111804    0.215397   0.048770        0.006697   
1                2.93   0.117959    0.020574   0.042005        0.009784   
2                1.38   0.160906    0.062674   0.174740        0.010851   
3                2.57   0.087152    0.005573   0.055984        0.010804   
4                0.93   0.153458    0.011764   0.064882        0.022295   

   T2020自家用車  ...  森林_ratio_valid  荒地_ratio_valid  建物用地_ratio_valid  \
0   0.351186  ...        0.665860        0.027077          0.188809   
1   0.633064  ..

In [3]:
#観光データ読み込み
tourism = pd.read_csv(
    "~/Desktop/趣味/city2022.csv",
    encoding="cp932"
)

print(tourism.head())
print(tourism.shape)
print(tourism.columns.tolist())

      年  月  地域区分   データ区分  都道府県コード 都道府県名  地域コード    地域名称      人数
0  2022  1  市区町村  観光来訪者数        1   北海道   1101  札幌市中央区  230903
1  2022  1  市区町村  観光来訪者数        1   北海道   1102   札幌市北区   12000
2  2022  1  市区町村  観光来訪者数        1   北海道   1103   札幌市東区   22333
3  2022  1  市区町村  観光来訪者数        1   北海道   1104  札幌市白石区    5575
4  2022  1  市区町村  観光来訪者数        1   北海道   1105  札幌市豊平区   22257
(22392, 9)
['年', '月', '地域区分', 'データ区分', '都道府県コード', '都道府県名', '地域コード', '地域名称', '人数']


In [4]:
#市を区にまとめる
def aggregate_city_name(name):
    # 例：札幌市中央区 → 札幌市
    # 例：横浜市中区 → 横浜市
    # 例：大阪市北区 → 大阪市
    if "市" in name and name.endswith("区"):
        return name.split("市")[0] + "市"
    else:
        return name

tourism["地域名称_集約"] = tourism["地域名称"].apply(aggregate_city_name)

tourism[["都道府県名", "地域名称", "地域名称_集約"]].head(20)

,都道府県名,地域名称,地域名称_集約
0,北海道,札幌市中央区,札幌市
1,北海道,札幌市北区,札幌市
2,北海道,札幌市東区,札幌市
3,北海道,札幌市白石区,札幌市
4,北海道,札幌市豊平区,札幌市
5,北海道,札幌市南区,札幌市
6,北海道,札幌市西区,札幌市
7,北海道,札幌市厚別区,札幌市
8,北海道,札幌市手稲区,札幌市
9,北海道,札幌市清田区,札幌市


In [5]:
#年間観光来訪者数を作る
tourism_year = tourism.groupby(
    ["都道府県名", "地域名称_集約"],
    as_index=False
)["人数"].sum()

tourism_year = tourism_year.rename(columns={
    "人数": "年間観光来訪者数"
})

tourism_year["地域"] = tourism_year["都道府県名"] + " " + tourism_year["地域名称_集約"]

tourism_year = tourism_year[["地域", "年間観光来訪者数"]]

print(tourism_year.head())
print(tourism_year.shape)

         地域  年間観光来訪者数
0  三重県 いなべ市    714717
1   三重県 亀山市   1098992
2   三重県 伊勢市   5486040
3   三重県 伊賀市   2210262
4  三重県 南伊勢町     44059
(1731, 2)


In [6]:
#元データと観光データを結合
drop_cols = [
    col for col in data.columns
    if "年間観光来訪者数" in col
    or "観光人気度" in col
    or "log_年間観光来訪者数" in col
]

data = data.drop(columns=drop_cols, errors="ignore")

# 観光データを結合
data = data.merge(
    tourism_year,
    on="地域",
    how="left"
)

# 欠損を0にする
data["年間観光来訪者数"] = data["年間観光来訪者数"].fillna(0)

print(data[["地域", "年間観光来訪者数"]].head())
print("欠損数:", data["年間観光来訪者数"].isna().sum())
print(data.shape)

        地域   年間観光来訪者数
0  北海道 札幌市  9513626.0
1  北海道 函館市  1423538.0
2  北海道 小樽市  1129466.0
3  北海道 旭川市  2339891.0
4  北海道 室蘭市   284144.0
欠損数: 0
(1728, 58)


In [7]:
#観光人気度を作る
def minmax(s):
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

# 観光客数は大きいためlog10をとる
data["log_年間観光来訪者数"] = np.log10(data["年間観光来訪者数"] + 1)

# 0〜100点にする
data["観光人気度"] = minmax(data["log_年間観光来訪者数"]) * 100

data[["地域", "年間観光来訪者数", "log_年間観光来訪者数", "観光人気度"]].sort_values(
    "年間観光来訪者数",
    ascending=False
).head(20)

,地域,年間観光来訪者数,log_年間観光来訪者数,観光人気度
973,愛知県 名古屋市,55145992.0,7.741514,100.000000
1101,大阪府 大阪市,34146093.0,7.533341,97.310953
616,東京都 千代田区,30760962.0,7.488000,96.725265
1075,京都府 京都市,27879311.0,7.445282,96.173463
618,東京都 港区,22475451.0,7.351708,94.964738
619,東京都 新宿区,22221546.0,7.346774,94.901002
631,東京都 豊島区,21668902.0,7.335837,94.759720
628,東京都 渋谷区,19852443.0,7.297814,94.268563
617,東京都 中央区,19206377.0,7.283445,94.082960
678,神奈川県 横浜市,19015393.0,7.279105,94.026897


In [8]:
#旅行向けスコアの作成
# =========================
# 基本特徴量
# =========================

pop_density = data["R2020_log10(10万人あたり人口密度)"]

building = data["建物用地_ratio_valid"]
road = data["道路_ratio_valid"]
rail_land = data["鉄道_ratio_valid"]

forest = data["森林_ratio_valid"]
rice = data["田_ratio_valid"]
farm = data["その他農用地_ratio_valid"]
river = data["河川湖沼_ratio_valid"]
beach = data["海浜_ratio_valid"]
sea = data["海水域_ratio_valid"]

walk = data["T2020徒歩だけ"]
train = data["T2020鉄道・電車"]
car = data["T2020自家用車"]

primary = data["J2020第1次産業就業者比率【％】"]

hotel_food = data["S2021_従業者数（民営）（宿泊業、飲食サービス業）"]
retail = data["S2021従業者数（民営）（小売業）"]
entertainment = data["S2021_従業者数（民営）（生活関連サービス業、娯楽業"]

In [9]:
#各スコアの作成
# =========================
# 旅行向けスコア
# =========================

data["都会度"] = (
    0.35 * minmax(pop_density)
    + 0.30 * minmax(building)
    + 0.15 * minmax(road)
    + 0.20 * minmax(train)
) * 100

data["自然度"] = (
    0.45 * minmax(forest)
    + 0.25 * minmax(rice + farm)
    + 0.15 * minmax(river)
    + 0.15 * (1 - minmax(building))
) * 100

data["海度"] = (
    0.70 * minmax(sea)
    + 0.30 * minmax(beach)
) * 100

data["山っぽさ"] = (
    0.70 * minmax(forest)
    + 0.20 * (1 - minmax(building))
    + 0.10 * (1 - minmax(pop_density))
) * 100

data["田舎度"] = (
    0.30 * minmax(forest)
    + 0.25 * minmax(rice + farm)
    + 0.20 * minmax(car)
    + 0.15 * minmax(primary)
    + 0.10 * (1 - minmax(pop_density))
) * 100

data["電車旅向き"] = (
    0.55 * minmax(train)
    + 0.20 * minmax(rail_land)
    + 0.15 * minmax(walk)
    + 0.10 * minmax(pop_density)
) * 100

data["車旅向き"] = (
    0.55 * minmax(car)
    + 0.20 * minmax(road)
    + 0.15 * (1 - minmax(train))
    + 0.10 * (1 - minmax(pop_density))
) * 100

data["観光・街歩き向き"] = (
    0.30 * minmax(hotel_food)
    + 0.25 * minmax(retail)
    + 0.20 * minmax(entertainment)
    + 0.15 * minmax(walk)
    + 0.10 * minmax(train)
) * 100

data["観光地らしさ"] = (
    0.25 * data["観光人気度"] / 100
    + 0.20 * data["海度"] / 100
    + 0.20 * data["山っぽさ"] / 100
    + 0.15 * data["自然度"] / 100
    + 0.10 * minmax(hotel_food)
    + 0.05 * minmax(entertainment)
    + 0.05 * minmax(retail)
) * 100

In [10]:
#確認
score_cols = [
    "都会度", "自然度", "海度", "山っぽさ", "田舎度",
    "電車旅向き", "車旅向き", "観光・街歩き向き",
    "観光人気度", "観光地らしさ"
]

data[["地域", "年間観光来訪者数"] + score_cols].head()

,地域,年間観光来訪者数,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き,観光人気度,観光地らしさ
0,北海道 札幌市,9513626.0,41.894755,43.859379,0.000000,65.357749,30.861230,30.530268,34.232558,19.059049,90.141879,45.248454
1,北海道 函館市,1423538.0,27.445808,53.257512,2.753736,80.739575,44.343092,12.390420,56.563040,19.077981,79.485349,48.366943
2,北海道 小樽市,1129466.0,28.739773,49.095775,5.045545,75.782821,38.111970,17.302367,46.428972,19.714833,78.187207,46.461454
3,北海道 旭川市,2339891.0,24.504167,48.408605,0.000000,63.762719,44.328090,9.260559,59.962465,15.762068,82.273259,43.754160
4,北海道 室蘭市,284144.0,34.186023,34.056736,9.043601,51.321299,34.136245,13.975297,60.070615,15.963786,70.445394,37.500902


In [11]:
data[["地域", "年間観光来訪者数", "観光人気度", "観光地らしさ", "海度", "山っぽさ", "自然度"]].sort_values(
    "観光地らしさ",
    ascending=False
).head(30)

,地域,年間観光来訪者数,観光人気度,観光地らしさ,海度,山っぽさ,自然度
1715,沖縄県 座間味村,41262.0,59.620906,61.748664,71.305906,79.023367,49.765015
677,東京都 小笠原村,19832.0,55.510956,57.676919,70.000000,83.506251,50.906998
1714,沖縄県 渡嘉敷村,13777.0,53.467399,56.893471,55.798309,81.379837,51.460065
1036,三重県 鳥羽市,2248085.0,82.048718,55.671335,15.983673,82.447658,53.757524
367,福島県 北塩原村,721386.0,75.672127,55.385531,0.000000,89.456370,57.945382
706,神奈川県 箱根町,3843379.0,85.057178,54.051054,0.000000,76.250200,50.376346
736,新潟県 湯沢町,1772521.0,80.715370,53.955837,0.000000,91.072575,57.137636
1725,沖縄県 竹富町,185241.0,68.045340,53.818213,12.300055,87.096550,56.529033
364,福島県 檜枝岐村,68025.0,62.425450,53.790119,0.000000,98.135321,59.268207
489,群馬県 片品村,1403726.0,79.406724,53.487526,0.000000,91.767012,57.965313


In [12]:


# 目的変数
y = data["log_年間観光来訪者数"]

# 説明変数
feature_cols = [
    "都会度",
    "自然度",
    "海度",
    "山っぽさ",
    "田舎度",
    "電車旅向き",
    "車旅向き",
    "観光・街歩き向き",
    "R2020_log10(10万人あたり人口密度)",
    "建物用地_ratio_valid",
    "道路_ratio_valid",
    "鉄道_ratio_valid",
    "森林_ratio_valid",
    "田_ratio_valid",
    "その他農用地_ratio_valid",
    "河川湖沼_ratio_valid",
    "海浜_ratio_valid",
    "海水域_ratio_valid",
    "S2021_従業者数（民営）（宿泊業、飲食サービス業）",
    "S2021従業者数（民営）（小売業）",
    "S2021_従業者数（民営）（生活関連サービス業、娯楽業"
]

X = data[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1382, 21)
X_test: (346, 21)
y_train: (1382,)
y_test: (346,)


In [13]:


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    return -scores.mean()

In [14]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best CV RMSE:", study.best_value)
print("Best params:")
print(study.best_params)

[I 2026-07-01 19:24:30,869] A new study created in memory with name: no-name-2ad4ce0d-8c39-46ac-9d5d-bf301486b093
[I 2026-07-01 19:24:38,987] Trial 0 finished with value: 0.7670723738781129 and parameters: {'n_estimators': 776, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.7670723738781129.
[I 2026-07-01 19:24:44,518] Trial 1 finished with value: 0.7607151928270632 and parameters: {'n_estimators': 775, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.7607151928270632.
[I 2026-07-01 19:24:50,204] Trial 2 finished with value: 0.7767959071957187 and parameters: {'n_estimators': 880, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7607151928270632.
[I 2026-07-01 19:24:51,428] Trial 3 finished with value: 0.7695602042616521 and parameters: {'n_estimators': 275, 'max_depth': 14, 'mi

Best CV RMSE: 0.7590113224610633
Best params:
{'n_estimators': 1000, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'log2'}


In [15]:
best_params = study.best_params

rf_best = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

rf_best.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamp

In [16]:
y_pred_train = rf_best.predict(X_train)
y_pred_test = rf_best.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("Random Forest after Optuna")
print("Train R2:", r2_train)
print("Test R2:", r2_test)
print("Test MAE:", mae_test)
print("Test RMSE:", rmse_test)

Random Forest after Optuna
Train R2: 0.7944260164178781
Test R2: 0.3480322003783537
Test MAE: 0.492223526186729
Test RMSE: 0.7681515163048199


In [17]:
data["ML予測_log観光来訪者数"] = rf_best.predict(X)

data["ML観光ポテンシャル"] = minmax(data["ML予測_log観光来訪者数"]) * 100

In [18]:
data[[
    "地域",
    "年間観光来訪者数",
    "観光人気度",
    "ML観光ポテンシャル",
    "都会度",
    "自然度",
    "海度",
    "山っぽさ",
    "観光・街歩き向き"
]].sort_values(
    "ML観光ポテンシャル",
    ascending=False
).head(30)

,地域,年間観光来訪者数,観光人気度,ML観光ポテンシャル,都会度,自然度,海度,山っぽさ,観光・街歩き向き
1101,大阪府 大阪市,34146093.0,97.310953,100.000000,71.728679,7.458713,2.992677,6.648348,17.839619
619,東京都 新宿区,22221546.0,94.901002,99.668078,84.233339,3.252535,0.000000,4.507652,21.670196
621,東京都 台東区,15404536.0,92.845517,96.315077,83.081530,4.676194,0.000000,5.267706,22.729419
622,東京都 墨田区,6593963.0,88.085439,96.042541,82.729353,6.102775,0.000000,4.503391,22.871969
618,東京都 港区,22475451.0,94.964738,95.052661,77.523811,6.493295,4.412439,8.985566,17.663796
623,東京都 江東区,16379234.0,93.189700,95.032030,72.987731,9.508879,10.701955,10.337366,16.972958
1075,京都府 京都市,27879311.0,96.173463,94.636387,41.777446,48.200471,0.000000,71.552746,19.851625
679,神奈川県 川崎市,11855845.0,91.376604,94.629278,71.588598,8.106835,6.188044,9.628527,20.857473
678,神奈川県 横浜市,19015393.0,94.026897,93.735835,69.307319,10.434729,2.762077,13.113854,20.947089
620,東京都 文京区,9696371.0,90.248617,93.245302,85.245816,3.795310,0.000000,5.216957,19.380667


In [19]:
data["観光ポテンシャル差"] = data["ML観光ポテンシャル"] - data["観光人気度"]

In [20]:
data[[
    "地域",
    "観光人気度",
    "ML観光ポテンシャル",
    "観光ポテンシャル差",
    "都会度",
    "自然度",
    "海度",
    "山っぽさ",
    "田舎度",
    "観光・街歩き向き"
]].sort_values(
    "観光ポテンシャル差",
    ascending=False
).head(30)

,地域,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,観光・街歩き向き
251,宮城県 塩竈市,0.000000,68.058153,68.058153,51.154719,16.086690,58.401762,25.094669,22.959432,16.829282
51,北海道 今金町,0.000000,62.206893,62.206893,6.753773,56.218200,0.000000,85.229302,55.788292,21.118370
123,北海道 訓子府町,0.000000,57.715861,57.715861,8.355039,52.096879,0.000000,63.379717,61.321630,11.662414
109,北海道 猿払村,0.000000,49.308559,49.308559,3.657830,51.951353,1.946474,80.009851,54.815359,9.974045
1117,大阪府 河内長野市,0.000000,34.508063,34.508063,38.069310,49.524705,0.000000,74.185791,36.006071,19.942288
401,茨城県 龍ケ崎市,0.000000,33.164819,33.164819,33.742727,32.803724,0.000000,26.287036,35.592322,17.410833
1288,島根県 知夫村,21.100116,53.899166,32.799050,14.687577,51.379401,26.958695,82.266521,48.506547,19.693587
588,千葉県 袖ケ浦市,0.000000,29.898211,29.898211,32.327999,32.923261,1.920391,35.117093,36.796225,11.283364
583,千葉県 鎌ケ谷市,0.000000,26.295818,26.295818,60.987430,13.012962,0.000000,12.019381,14.549123,24.103919
236,岩手県 金ケ崎町,0.000000,26.072396,26.072396,14.563535,50.296200,0.000000,62.016848,54.515972,6.782429


In [21]:
def recommend_area_ml(
    data,
    urban=0,
    nature=0,
    sea=0,
    mountain=0,
    rural=0,
    train_trip=0,
    car_trip=0,
    sightseeing=0,
    ml_tourism=0.5,
    top_n=10
):
    result = data.copy()

    total_weight = (
        urban + nature + sea + mountain + rural
        + train_trip + car_trip + sightseeing + ml_tourism
    )

    if total_weight == 0:
        raise ValueError("少なくとも1つの条件に重みを入れてください")

    result["おすすめスコア"] = (
        urban * result["都会度"]
        + nature * result["自然度"]
        + sea * result["海度"]
        + mountain * result["山っぽさ"]
        + rural * result["田舎度"]
        + train_trip * result["電車旅向き"]
        + car_trip * result["車旅向き"]
        + sightseeing * result["観光・街歩き向き"]
        + ml_tourism * result["ML観光ポテンシャル"]
    ) / total_weight

    display_cols = [
        "地域",
        "おすすめスコア",
        "観光人気度",
        "ML観光ポテンシャル",
        "観光ポテンシャル差",
        "都会度",
        "自然度",
        "海度",
        "山っぽさ",
        "田舎度",
        "電車旅向き",
        "車旅向き",
        "観光・街歩き向き"
    ]

    return result.sort_values("おすすめスコア", ascending=False)[display_cols].head(top_n)

In [22]:
recommend_area_ml(
    data,
    sea=1.0,
    nature=0.8,
    ml_tourism=0.6,
    top_n=20
)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
1715,沖縄県 座間味村,55.958239,59.620906,38.636424,-20.984483,15.646162,49.765015,71.305906,79.023367,34.783201,19.068919,35.183567,46.431111
677,東京都 小笠原村,53.561593,55.510956,29.703710,-25.807247,7.895623,50.906998,70.000000,83.506251,39.540315,14.819389,38.547114,32.110332
1714,沖縄県 渡嘉敷村,51.995438,53.467399,46.371149,-7.096251,16.956179,51.460065,55.798309,81.379837,40.513268,15.302928,47.738504,40.291104
1394,香川県 直島町,51.460717,67.364017,45.981705,-21.382312,22.905439,38.554626,65.072997,60.254601,31.486121,8.911615,45.414110,15.602115
1552,長崎県 小値賀町,51.059604,58.995489,48.841153,-10.154336,15.375095,45.923726,56.499377,61.463968,48.796648,9.301229,57.202855,25.468232
1717,沖縄県 渡名喜村,49.882809,37.500287,19.819627,-17.680660,16.044251,46.545180,70.590820,71.718261,36.971794,17.526315,38.631116,35.748825
1248,和歌山県 太地町,49.190147,70.299770,62.475672,-7.824097,27.819326,44.059684,45.323202,67.952244,43.599677,10.498559,64.026717,18.093428
1355,山口県 上関町,47.540241,67.651913,56.799773,-10.852140,14.532707,54.842662,36.142586,83.908547,51.706770,8.413006,61.832560,15.542171
1613,大分県 姫島村,46.754400,41.787991,42.734052,0.946061,20.069094,45.767200,49.956368,67.784787,37.126235,8.360846,38.719217,23.994002
251,宮城県 塩竈市,46.710836,0.000000,68.058153,68.058153,51.154719,16.086690,58.401762,25.094669,22.959432,25.183021,53.760720,16.829282


In [23]:
recommend_area_ml(
    data,
    mountain=1.0,
    nature=1.0,
    rural=0.5,
    ml_tourism=0.6,
    top_n=20
)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
914,岐阜県 郡上市,72.145425,84.332062,81.821013,-2.511049,17.426931,57.977010,0.000000,89.706491,53.749420,7.623474,70.102357,16.260495
489,群馬県 片品村,71.907456,79.406724,73.781537,-5.625187,11.963259,57.965313,0.000000,91.767012,57.823730,5.235531,71.588853,23.572944
492,群馬県 みなかみ町,71.853213,83.409166,81.549065,-1.860101,12.406199,56.884255,0.000000,89.672830,54.516874,8.979206,69.059534,22.370753
1048,三重県 大台町,71.733124,74.138708,76.774770,2.636062,17.342811,58.454599,0.000000,91.368575,52.969295,9.868467,66.505256,15.265492
895,長野県 栄村,71.712841,68.186314,63.301640,-4.884674,8.964725,59.123379,0.000000,94.549951,61.310986,7.797960,69.928061,18.092004
898,岐阜県 高山市,71.706554,83.101092,77.003661,-6.097431,16.202897,58.358761,0.000000,90.892143,53.674435,7.301005,66.880051,18.295927
217,岩手県 宮古市,71.629326,76.644527,74.756176,-1.888352,17.697850,58.905549,1.279312,92.007140,52.569036,8.430994,65.327241,16.744611
264,宮城県 七ヶ宿町,71.612384,69.486245,64.384990,-5.101255,6.686306,59.119069,0.000000,94.645609,59.205436,5.311687,71.120038,17.997724
912,岐阜県 飛騨市,71.600367,75.624363,73.751055,-1.873308,16.909548,58.873579,0.000000,91.784730,54.104392,7.636219,68.721806,12.468032
235,岩手県 西和賀町,71.592972,69.348399,66.376297,-2.972102,7.634018,58.958013,0.000000,93.281340,59.746163,4.722357,73.197012,16.249505


In [24]:
recommend_area_ml(
    data,
    urban=1.0,
    sightseeing=0.8,
    train_trip=0.6,
    ml_tourism=0.5,
    top_n=20
)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
631,東京都 豊島区,68.354470,94.759720,90.357779,-4.401941,87.040325,1.256182,0.000000,1.714226,0.159573,75.271087,5.669264,26.057621
624,東京都 品川区,66.282245,85.482055,91.331228,5.849173,80.128374,5.177418,5.007457,7.004974,0.994692,85.463277,8.053139,18.933195
619,東京都 新宿区,66.225917,94.901002,99.668078,4.767077,84.233339,3.252535,0.000000,4.507652,1.145125,67.752706,9.479467,21.670196
622,東京都 墨田区,65.272795,88.085439,96.042541,7.957102,82.729353,6.102775,0.000000,4.503391,0.642696,67.071512,11.282865,22.871969
621,東京都 台東区,65.215572,92.845517,96.315077,3.469560,83.081530,4.676194,0.000000,5.267706,0.824416,66.170925,11.862631,22.729419
620,東京都 文京区,64.818263,90.248617,93.245302,2.996685,85.245816,3.795310,0.000000,5.216957,1.375475,67.666603,8.660916,19.380667
629,東京都 中野区,64.151363,82.306517,87.837468,5.530951,84.121688,0.705612,0.000000,0.516878,0.369492,65.954137,2.437615,23.032562
633,東京都 荒川区,63.842233,78.721080,85.550002,6.828922,79.249681,4.299360,0.000000,3.576983,0.495341,75.081958,4.909424,22.585774
630,東京都 杉並区,63.616936,75.048266,82.510686,7.462419,83.625573,1.582599,0.000000,2.126077,1.062103,64.892072,5.830995,25.841194
625,東京都 目黒区,63.412393,81.973937,85.912216,3.938279,83.223185,0.813168,0.000000,1.058706,0.888461,62.034285,6.085769,25.620094


In [25]:
#区をなくす
data_no_ku = data[~data["地域"].str.endswith("区")].copy()

In [26]:
recommend_area_ml(
    data_no_ku,
    urban=1.0,
    sightseeing=0.8,
    train_trip=0.6,
    ml_tourism=0.5,
    top_n=20
)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
641,東京都 武蔵野市,59.342274,78.844490,76.684867,-2.159623,76.369572,2.168664,0.000000,3.290575,1.911392,58.259783,6.496057,28.030901
679,神奈川県 川崎市,58.195259,91.376604,94.629278,3.252675,71.588598,8.106835,6.188044,9.628527,4.473478,55.295058,11.789775,20.857473
1101,大阪府 大阪市,57.176180,97.310953,100.000000,2.689047,71.728679,7.458713,2.992677,6.648348,2.458827,49.684247,17.478614,17.839619
518,埼玉県 蕨市,56.511498,61.680421,73.786098,12.105677,77.546174,0.259846,0.000000,0.994944,2.515007,50.188537,11.762708,24.163748
1106,大阪府 吹田市,56.474788,82.760571,88.736459,5.975888,71.917950,8.250151,0.000000,12.035592,5.582625,52.297123,20.227644,20.140541
646,東京都 調布市,56.440863,78.008539,79.315517,1.306978,72.965655,7.045107,0.000000,7.021336,3.818078,52.691707,10.210011,24.300081
586,千葉県 浦安市,56.286136,91.373950,91.958478,0.584528,68.609380,6.766621,11.919997,8.432870,2.661598,41.661213,22.427893,29.555561
678,神奈川県 横浜市,56.137231,94.026897,93.735835,-0.291061,69.307319,10.434729,2.762077,13.113854,8.206345,49.775105,17.155889,20.947089
655,東京都 狛江市,55.509055,64.166688,66.567116,2.400429,75.488416,3.856030,0.000000,2.257912,2.135794,54.772833,7.069253,24.175733
564,千葉県 市川市,55.243596,76.941983,80.460259,3.518276,69.515601,10.365815,2.380854,8.816729,6.852105,53.217545,14.586093,23.162713


In [27]:
#旅行タイプのリセット機能
travel_patterns_ml = {
    "海と自然でのんびり": {
        "sea": 1.0,
        "nature": 0.8,
        "ml_tourism": 0.6
    },
    "海沿い観光・リゾート": {
        "sea": 1.0,
        "sightseeing": 0.8,
        "ml_tourism": 0.7
    },
    "山と自然を楽しむ": {
        "mountain": 1.0,
        "nature": 1.0,
        "rural": 0.5,
        "ml_tourism": 0.6
    },
    "田舎でのんびり": {
        "rural": 1.0,
        "nature": 0.8,
        "car_trip": 0.6,
        "ml_tourism": 0.4
    },
    "都会で街歩き": {
        "urban": 1.0,
        "sightseeing": 0.8,
        "train_trip": 0.6,
        "ml_tourism": 0.5
    },
    "電車で行ける観光地": {
        "train_trip": 1.0,
        "sightseeing": 0.8,
        "urban": 0.4,
        "ml_tourism": 0.6
    }
}

In [28]:
#タイプ名で推薦する関数
def recommend_by_pattern_ml(data, pattern_name, top_n=20):
    if pattern_name not in travel_patterns_ml:
        raise ValueError(f"使える旅行タイプ: {list(travel_patterns_ml.keys())}")

    params = travel_patterns_ml[pattern_name]

    return recommend_area_ml(
        data,
        top_n=top_n,
        **params
    )

In [29]:
recommend_by_pattern_ml(data, "海と自然でのんびり", top_n=20)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
1715,沖縄県 座間味村,55.958239,59.620906,38.636424,-20.984483,15.646162,49.765015,71.305906,79.023367,34.783201,19.068919,35.183567,46.431111
677,東京都 小笠原村,53.561593,55.510956,29.703710,-25.807247,7.895623,50.906998,70.000000,83.506251,39.540315,14.819389,38.547114,32.110332
1714,沖縄県 渡嘉敷村,51.995438,53.467399,46.371149,-7.096251,16.956179,51.460065,55.798309,81.379837,40.513268,15.302928,47.738504,40.291104
1394,香川県 直島町,51.460717,67.364017,45.981705,-21.382312,22.905439,38.554626,65.072997,60.254601,31.486121,8.911615,45.414110,15.602115
1552,長崎県 小値賀町,51.059604,58.995489,48.841153,-10.154336,15.375095,45.923726,56.499377,61.463968,48.796648,9.301229,57.202855,25.468232
1717,沖縄県 渡名喜村,49.882809,37.500287,19.819627,-17.680660,16.044251,46.545180,70.590820,71.718261,36.971794,17.526315,38.631116,35.748825
1248,和歌山県 太地町,49.190147,70.299770,62.475672,-7.824097,27.819326,44.059684,45.323202,67.952244,43.599677,10.498559,64.026717,18.093428
1355,山口県 上関町,47.540241,67.651913,56.799773,-10.852140,14.532707,54.842662,36.142586,83.908547,51.706770,8.413006,61.832560,15.542171
1613,大分県 姫島村,46.754400,41.787991,42.734052,0.946061,20.069094,45.767200,49.956368,67.784787,37.126235,8.360846,38.719217,23.994002
251,宮城県 塩竈市,46.710836,0.000000,68.058153,68.058153,51.154719,16.086690,58.401762,25.094669,22.959432,25.183021,53.760720,16.829282


In [30]:
recommend_by_pattern_ml(data_no_ku, "都会で街歩き", top_n=20)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
641,東京都 武蔵野市,59.342274,78.844490,76.684867,-2.159623,76.369572,2.168664,0.000000,3.290575,1.911392,58.259783,6.496057,28.030901
679,神奈川県 川崎市,58.195259,91.376604,94.629278,3.252675,71.588598,8.106835,6.188044,9.628527,4.473478,55.295058,11.789775,20.857473
1101,大阪府 大阪市,57.176180,97.310953,100.000000,2.689047,71.728679,7.458713,2.992677,6.648348,2.458827,49.684247,17.478614,17.839619
518,埼玉県 蕨市,56.511498,61.680421,73.786098,12.105677,77.546174,0.259846,0.000000,0.994944,2.515007,50.188537,11.762708,24.163748
1106,大阪府 吹田市,56.474788,82.760571,88.736459,5.975888,71.917950,8.250151,0.000000,12.035592,5.582625,52.297123,20.227644,20.140541
646,東京都 調布市,56.440863,78.008539,79.315517,1.306978,72.965655,7.045107,0.000000,7.021336,3.818078,52.691707,10.210011,24.300081
586,千葉県 浦安市,56.286136,91.373950,91.958478,0.584528,68.609380,6.766621,11.919997,8.432870,2.661598,41.661213,22.427893,29.555561
678,神奈川県 横浜市,56.137231,94.026897,93.735835,-0.291061,69.307319,10.434729,2.762077,13.113854,8.206345,49.775105,17.155889,20.947089
655,東京都 狛江市,55.509055,64.166688,66.567116,2.400429,75.488416,3.856030,0.000000,2.257912,2.135794,54.772833,7.069253,24.175733
564,千葉県 市川市,55.243596,76.941983,80.460259,3.518276,69.515601,10.365815,2.380854,8.816729,6.852105,53.217545,14.586093,23.162713


In [31]:
#観光ポテンシャル差ランキング（観光地としての特徴はあるが観光客が少ない）
data[[
    "地域",
    "観光人気度",
    "ML観光ポテンシャル",
    "観光ポテンシャル差",
    "都会度",
    "自然度",
    "海度",
    "山っぽさ",
    "田舎度",
    "観光・街歩き向き"
]].sort_values(
    "観光ポテンシャル差",
    ascending=False
).head(30)

,地域,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,観光・街歩き向き
251,宮城県 塩竈市,0.000000,68.058153,68.058153,51.154719,16.086690,58.401762,25.094669,22.959432,16.829282
51,北海道 今金町,0.000000,62.206893,62.206893,6.753773,56.218200,0.000000,85.229302,55.788292,21.118370
123,北海道 訓子府町,0.000000,57.715861,57.715861,8.355039,52.096879,0.000000,63.379717,61.321630,11.662414
109,北海道 猿払村,0.000000,49.308559,49.308559,3.657830,51.951353,1.946474,80.009851,54.815359,9.974045
1117,大阪府 河内長野市,0.000000,34.508063,34.508063,38.069310,49.524705,0.000000,74.185791,36.006071,19.942288
401,茨城県 龍ケ崎市,0.000000,33.164819,33.164819,33.742727,32.803724,0.000000,26.287036,35.592322,17.410833
1288,島根県 知夫村,21.100116,53.899166,32.799050,14.687577,51.379401,26.958695,82.266521,48.506547,19.693587
588,千葉県 袖ケ浦市,0.000000,29.898211,29.898211,32.327999,32.923261,1.920391,35.117093,36.796225,11.283364
583,千葉県 鎌ケ谷市,0.000000,26.295818,26.295818,60.987430,13.012962,0.000000,12.019381,14.549123,24.103919
236,岩手県 金ケ崎町,0.000000,26.072396,26.072396,14.563535,50.296200,0.000000,62.016848,54.515972,6.782429


In [32]:
data.to_csv(
    "~/Desktop/趣味/travel_recommendation_data_ml.csv",
    index=False,
    encoding="utf-8-sig"
)

print("保存しました")

保存しました


In [33]:
data = pd.read_csv("~/Desktop/趣味/travel_recommendation_data_ml.csv")

In [34]:
recommend_area_ml(
    data,
    sea=1.0,
    nature=0.8,
    ml_tourism=0.6,
    top_n=20
)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
1715,沖縄県 座間味村,55.958239,59.620906,38.636424,-20.984483,15.646162,49.765015,71.305906,79.023367,34.783201,19.068919,35.183567,46.431111
677,東京都 小笠原村,53.561593,55.510956,29.703710,-25.807247,7.895623,50.906998,70.000000,83.506251,39.540315,14.819389,38.547114,32.110332
1714,沖縄県 渡嘉敷村,51.995438,53.467399,46.371149,-7.096251,16.956179,51.460065,55.798309,81.379837,40.513268,15.302928,47.738504,40.291104
1394,香川県 直島町,51.460717,67.364017,45.981705,-21.382312,22.905439,38.554626,65.072997,60.254601,31.486121,8.911615,45.414110,15.602115
1552,長崎県 小値賀町,51.059604,58.995489,48.841153,-10.154336,15.375095,45.923726,56.499377,61.463968,48.796648,9.301229,57.202855,25.468232
1717,沖縄県 渡名喜村,49.882809,37.500287,19.819627,-17.680660,16.044251,46.545180,70.590820,71.718261,36.971794,17.526315,38.631116,35.748825
1248,和歌山県 太地町,49.190147,70.299770,62.475672,-7.824097,27.819326,44.059684,45.323202,67.952244,43.599677,10.498559,64.026717,18.093428
1355,山口県 上関町,47.540241,67.651913,56.799773,-10.852140,14.532707,54.842662,36.142586,83.908547,51.706770,8.413006,61.832560,15.542171
1613,大分県 姫島村,46.754400,41.787991,42.734052,0.946061,20.069094,45.767200,49.956368,67.784787,37.126235,8.360846,38.719217,23.994002
251,宮城県 塩竈市,46.710836,0.000000,68.058153,68.058153,51.154719,16.086690,58.401762,25.094669,22.959432,25.183021,53.760720,16.829282


In [35]:
travel_patterns_ml = {
    "海と自然でのんびり": {
        "sea": 1.0,
        "nature": 0.8,
        "ml_tourism": 0.6
    },
    "海沿い観光・リゾート": {
        "sea": 1.0,
        "sightseeing": 0.8,
        "ml_tourism": 0.7
    },
    "山と自然を楽しむ": {
        "mountain": 1.0,
        "nature": 1.0,
        "rural": 0.5,
        "ml_tourism": 0.6
    },
    "田舎でのんびり": {
        "rural": 1.0,
        "nature": 0.8,
        "car_trip": 0.6,
        "ml_tourism": 0.4
    },
    "都会で街歩き": {
        "urban": 1.0,
        "sightseeing": 0.8,
        "train_trip": 0.6,
        "ml_tourism": 0.5
    },
    "電車で行ける観光地": {
        "train_trip": 1.0,
        "sightseeing": 0.8,
        "urban": 0.4,
        "ml_tourism": 0.6
    }
}

In [36]:
def recommend_by_pattern_ml(data, pattern_name, top_n=20):
    if pattern_name not in travel_patterns_ml:
        raise ValueError(f"使える旅行タイプ: {list(travel_patterns_ml.keys())}")

    params = travel_patterns_ml[pattern_name]

    return recommend_area_ml(
        data,
        top_n=top_n,
        **params
    )

In [37]:
recommend_by_pattern_ml(data, "海と自然でのんびり", top_n=5)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
1715,沖縄県 座間味村,55.958239,59.620906,38.636424,-20.984483,15.646162,49.765015,71.305906,79.023367,34.783201,19.068919,35.183567,46.431111
677,東京都 小笠原村,53.561593,55.510956,29.703710,-25.807247,7.895623,50.906998,70.000000,83.506251,39.540315,14.819389,38.547114,32.110332
1714,沖縄県 渡嘉敷村,51.995438,53.467399,46.371149,-7.096251,16.956179,51.460065,55.798309,81.379837,40.513268,15.302928,47.738504,40.291104
1394,香川県 直島町,51.460717,67.364017,45.981705,-21.382312,22.905439,38.554626,65.072997,60.254601,31.486121,8.911615,45.414110,15.602115
1552,長崎県 小値賀町,51.059604,58.995489,48.841153,-10.154336,15.375095,45.923726,56.499377,61.463968,48.796648,9.301229,57.202855,25.468232


In [38]:
#簡易診断
def travel_diagnosis_ml(data, top_n=10):
    print("旅行タイプを選んでください")
    print("====================")

    pattern_names = list(travel_patterns_ml.keys())

    for i, name in enumerate(pattern_names, start=1):
        print(f"{i}. {name}")

    choice = int(input("番号を入力してください: "))

    if choice < 1 or choice > len(pattern_names):
        print("正しい番号を入力してください")
        return None

    selected_pattern = pattern_names[choice - 1]

    print(f"\n選択した旅行タイプ: {selected_pattern}")
    print("====================")

    return recommend_by_pattern_ml(data, selected_pattern, top_n=top_n)